# Лабораторная работа №1
## Основы Python через анализ рыночной позиции

### Итог работы

Вы получите набор исходных строк из условной торговой системы,
преобразуете их в корректные типы, рассчитаете экономику позиции,
проверите риск-ограничения и сформируете текстовый отчёт. Все задачи
образуют один конвейер, а не набор несвязанных упражнений.

### Разрешённые средства

Переменные, скалярные типы, преобразование типов, арифметические и
логические операторы, методы строк, f-строки, `None` и условное
выражение `x if condition else y`. Циклы, функции, классы, NumPy и
Pandas пока не используются.

### Правило выполнения

Запускайте ячейки сверху вниз.



## 1. Состояние ноутбука и переменные

Jupyter хранит результаты выполненных ячеек. Если изменить значение
выше, но не перезапустить зависимые ячейки, на экране может остаться
устаревший результат. Поэтому перед сдачей используйте **Restart
Kernel and Run All Cells**.

Переменная — имя, связанное с объектом. Оператор `=` выполняет
присваивание, `==` — сравнение.


In [1]:
import sys

print("Python:", sys.version.split()[0])
position_state = "draft"
print("Начальное состояние:", position_state)


Python: 3.12.1
Начальное состояние: draft


### Задание 1. Паспорт выполнения

Создайте переменные:

- `student_name` — непустая строка с вашим именем;
- `lab_number` — целое число 1;
- `is_completed` — логическое `False`;
- `submitted_at` — `None`, так как работа ещё не сдана.

Выведите значения и их типы через `type()`.


In [2]:
student_name = "Руслан"  # впишите имя
lab_number = 1
is_completed = False
submitted_at = None

print(student_name, type(student_name))
print(lab_number, type(lab_number))
print(is_completed, type(is_completed))
print(submitted_at, type(submitted_at))


Руслан <class 'str'>
1 <class 'int'>
False <class 'bool'>
None <class 'NoneType'>


## 2. Граница системы: данные приходят строками

CSV, формы и API нередко передают числа как текст. Строка `"120"`
и число `120` — разные объекты: строку нельзя безопасно использовать
в арифметике до явного преобразования.


In [3]:
# Снимок позиции в том виде, в котором его прислала внешняя система.
raw_symbol = "  sber "
raw_quantity = "120"
raw_entry_price = "245.50"
raw_current_price = "258.20"
raw_dividend_per_share = "3.40"
raw_commission_percent = "0.05"  # именно проценты: 0.05%
raw_is_long = "yes"
raw_market_status = " OPEN "


### Задание 2. Приведение к рабочим типам

Создайте нормализованные значения:

- `symbol`: строка без внешних пробелов в верхнем регистре;
- `quantity`: `int`;
- цены и дивиденд: `float`;
- `commission_rate`: доля, то есть 0.0005, а не 0.05;
- `is_long`: результат сравнения нормализованной строки с `"yes"`;
- `market_status`: верхний регистр без внешних пробелов.

Исходные `raw_*` переменные не изменяйте.


In [4]:
symbol =raw_symbol.strip().upper()
quantity = int(raw_quantity)
entry_price = float(raw_entry_price)
current_price = float(raw_current_price)
dividend_per_share = float(raw_dividend_per_share)
commission_rate = float(raw_commission_percent)/100
is_long = raw_is_long.strip().lower() == "yes"
market_status = raw_market_status.strip().upper()

print(symbol, quantity, entry_price, current_price, dividend_per_share, commission_rate, is_long, market_status)


SBER 120 245.5 258.2 3.4 0.0005 True OPEN


## 3. Числа, единицы измерения и порядок операций

В одном выражении легко смешать рубли, штуки и проценты. Перед
расчётом подпишите смысл переменных:

- `quantity × price` → рубли;
- `turnover × commission_rate` → рубли;
- `pnl / invested_capital` → безразмерная доля.

Скобки — часть модели, а не косметика. Для сравнения результатов
`float` используйте допустимую погрешность, а не всегда точное `==`.


### Задание 3. Экономика позиции

Рассчитайте:

1. `entry_turnover` — стоимость покупки без комиссии;
2. `entry_commission`;
3. `invested_capital` — покупка вместе с комиссией;
4. `current_value`;
5. `dividend_income`;
6. `gross_pnl` без дивидендов и комиссии;
7. `net_pnl` с дивидендами и комиссией;
8. `return_rate` относительно вложенного капитала;
9. `price_change_rate`;
10. `break_even_price` с учётом дивидендов и комиссии.


In [5]:
entry_turnover = quantity*entry_price # сумма покупки акций
entry_commission = entry_turnover*commission_rate #комиссия брокера
invested_capital = entry_turnover+entry_commission # сколько всего потрачено
current_value = quantity*current_price # текущая стоимость пакета акций
dividend_income = quantity*dividend_per_share # суммарные дивиденды
gross_pnl = current_value-entry_turnover #грязная прибыль
net_pnl = gross_pnl+dividend_income-entry_commission #чистая прибыль
return_rate = net_pnl/invested_capital # доходность позиции
price_change_rate = (current_price- entry_price)/entry_price #процент изменения цены акции
break_even_price = (invested_capital - dividend_income)/quantity # цена безубыточности
# YOUR CODE HERE

print("entry_turnover:"     , entry_turnover)
print("entry_commission:"   , entry_commission)
print("invested_capital:"   , invested_capital)
print("current_value:"      , current_value)
print("dividend_income:"    , dividend_income)
print("gross_pnl:"          , gross_pnl)
print("net_pnl:"            , net_pnl)
print("return_rate:"        , return_rate)
print("price_change_rate:"  , price_change_rate)
print("break_even_price:"   , break_even_price)


entry_turnover: 29460.0
entry_commission: 14.73
invested_capital: 29474.73
current_value: 30984.0
dividend_income: 408.0
gross_pnl: 1524.0
net_pnl: 1917.27
return_rate: 0.06504792410312156
price_change_rate: 0.0517311608961303
break_even_price: 242.22275


### Задание 4. Время торговой сессии

Время задано строками `HH:MM:SS`. Получите часы, минуты и секунды
срезами строки, переведите начало и конец в секунды от полуночи.
Рассчитайте длительность сессии, число сделок в час и средний оборот
в минуту. Сессия заканчивается в тот же день.


In [6]:
session_start = "09:50:00"
session_end = "18:50:00"
executed_trades = 427
session_turnover = 78_450_000.0


In [7]:
start_seconds = ( int(session_start[0:2]) * 3600 
                + int(session_start[3:5]) * 60 
                + int(session_start[6:8])
                )
end_seconds =   ( int(session_end[0:2]) * 3600 
                + int(session_end[3:5]) * 60 
                + int(session_end[6:8])
                ) 
session_seconds = end_seconds-start_seconds
session_hours = session_seconds/3600
trades_per_hour = executed_trades/session_hours
turnover_per_minute = session_turnover/(session_seconds/60)
# YOUR CODE HERE

print("session_seconds:", session_seconds)
print("session_hours:", session_hours)
print("trades_per_hour:", trades_per_hour)
print("turnover_per_minute:", turnover_per_minute)


session_seconds: 32400
session_hours: 9.0
trades_per_hour: 47.44444444444444
turnover_per_minute: 145277.77777777778


## 4. Строки как часть интерфейса

Внутри программы данные должны иметь единый формат. На границе
системы из них создают человекочитаемые подписи и машинные ID.
`strip`, `lower`, `upper`, `title`, `split`, `join` и f-строки
покрывают большую часть задач первой обработки текста.


### Задание 5. Идентификаторы и подписи

Нормализуйте имя клиента, создайте email и идентификатор позиции.

- имя: `Ivan Sergeevich Petrov`;
- login: `ivan.petrov`;
- email: `ivan.petrov@example.edu`;
- instrument: `MOEX:TQBR:SBER`;
- position_id: `MOEX-TQBR-SBER-000120`.


In [ ]:
raw_client_name = "  IVAN   sergeevich   PETROV  "
market = "moex"
board = "tqbr"
id = 120
first_name, lat_name,mid_name = raw_client_name.strip().lower().split()
client_name = f"{first_name.title()} {lat_name.title()} {mid_name.title()}"
name_parts = raw_client_name.strip().lower().split()
login = f"{first_name}.{mid_name}"
email = f"{login}@example.edu"
instrument = f"{market.upper()}:{board.upper()}:{symbol.upper()}"
position_id =  f"{market.upper()}-{board.upper()}-{symbol.upper()}-{id:06d}"
# YOUR CODE HERE

print("client_name:",   client_name)
print("login:",         login)
print("email:",         email)
print("instrument:",    instrument)
print("position_id:",   position_id)


client_name: Ivan Sergeevich Petrov
login: ivan.petrov
email: ivan.petrov@example.edu
instrument: MOEX:TQBR:SBER
position_id: MOEX-TQBR-SBER-000120


## 5. Логика, ограничения и отсутствующие данные

Риск-решение складывается из независимых проверок. Храните каждую
проверку в отдельной переменной: так проще объяснить отказ. `None`
означает отсутствие значения, а не ноль и не пустую строку.


### Задание 6. Риск-фильтр

Проверки:

- стоимость позиции не выше 40 000 RUB;
- доходность не ниже −7%;
- среднедневной объём не ниже 500 000 бумаг;
- рынок открыт;
- метка времени риск-данных присутствует.

Сначала оставьте метку `None`. Решение должно быть `REFRESH_DATA`.
После этого задайте метку времени строкой и пересчитайте решение:
оно должно стать `ALLOW`.


In [ ]:
max_position_value = 40_000.0
max_loss_rate = 0.07
min_daily_volume = 500_000
average_daily_volume = 1_250_000
risk_data_timestamp = None

position_limit_ok =  current_value <= max_position_value
loss_limit_ok =  return_rate >= -max_loss_rate
liquidity_ok = average_daily_volume >= min_daily_volume
market_open = market_status == "OPEN"
data_complete =  risk_data_timestamp is not None
risk_checks_pass = (
    position_limit_ok and 
    loss_limit_ok     and 
    liquidity_ok      and 
    market_open       and 
    data_complete
    )
decision_without_data = data_complete
decision_without_data = ("ALLOW"    if risk_checks_pass 
                                    else "REFRESH_DATA")

# После первого решения задайте непустую метку времени.
risk_data_timestamp = "2004-03-01T13:00:00"
data_complete = risk_data_timestamp is not None
final_risk_checks_pass = (
                            position_limit_ok and 
                            loss_limit_ok and 
                            liquidity_ok and 
                            market_open and 
                            data_complete
                        )
decision_with_data = ("ALLOW" if final_risk_checks_pass 
                                else "REFRESH_DATA")
# YOUR CODE HERE

print("decision_without_data:", decision_without_data)
print("decision_with_data:", decision_with_data)


False
decision_without_data: REFRESH_DATA
decision_with_data: ALLOW


## 6. Отладка: код выполняется не так, как задумано

Ошибка бывает синтаксической, типовой или логической. Последняя
опаснее: программа работает, но считает неверно. Прочитайте пять
проблем ниже и исправьте их, не меняя исходные `raw_*` значения.

### Задание 7. Исправление ошибок

1. `"120" * 3` повторяет строку, а не умножает число.
2. `0.05` процента ошибочно используют как долю 5%.
3. В имени остаются повторяющиеся пробелы.
4. `0.1 + 0.2 == 0.3` ненадёжно для float.
5. Отсутствие данных ошибочно проверяют через `== "None"`.


In [10]:
fixed_triple_quantity = int(raw_quantity) * 3
fixed_commission = (int(raw_quantity) * 
                    float(raw_entry_price) * 
                    (float(raw_commission_percent) / 100))
fixed_client_name = " ".join(raw_client_name.title().split())
float_values_equal = abs((0.1 + 0.2) - 0.3) < 1e-9
missing_value_detected = submitted_at is None
# YOUR CODE HERE


In [11]:
assert fixed_triple_quantity == 360
assert abs(fixed_commission - 14.73) < 1e-9
assert fixed_client_name == "Ivan Sergeevich Petrov"
assert float_values_equal is True
assert missing_value_detected is True
print("Задание 7: OK")


Задание 7: OK


## 7. Итоговый мини-проект

Ниже находится новый, независимый снимок позиции. Не копируйте
готовые числа из предыдущих ячеек: повторите весь путь от строк до
решения.

### Задание 8. Позиция GAZP и стресс-сценарий

1. Преобразуйте исходные данные.
2. Рассчитайте вложенный капитал, текущую стоимость, чистый PnL и
   доходность.
3. Проверьте лимит позиции, максимальный убыток, ликвидность,
   статус рынка и наличие метки риска.
4. Сформируйте строку отчёта строго заданного формата.
5. Проведите стресс-сценарий с ценой 172 RUB и получите новое
   решение без изменения базового результата.


In [12]:
final_raw_symbol = " gazp "
final_raw_quantity = "350"
final_raw_entry_price = "181.40"
final_raw_current_price = "176.90"
final_raw_dividend = "0"
final_raw_commission_percent = "0.04"
final_market_status = "open"
final_raw_daily_volume = "850000"
final_risk_timestamp = "2026-03-01T10:05:00"

final_max_position_value = 70_000.0
final_max_loss_rate = 0.04
final_min_daily_volume = 500_000
stress_price = 172.0


In [ ]:
final_symbol = final_raw_symbol.strip().upper()
final_quantity = int(final_raw_quantity)
final_entry_price = float(final_raw_entry_price)
final_current_price = float(final_raw_current_price)
final_dividend = float(final_raw_dividend)
final_commission_rate = float(final_raw_commission_percent) / 100
final_daily_volume = int(final_raw_daily_volume)

final_entry_turnover = final_quantity*final_entry_price
final_entry_commission = final_entry_turnover*final_commission_rate
final_invested_capital = final_entry_turnover+final_entry_commission
final_current_value = final_quantity*final_current_price
final_net_pnl = (
    final_current_value 
    - final_invested_capital 
    + (final_quantity*final_dividend)
)
final_return_rate = final_net_pnl/final_invested_capital

final_position_ok = final_current_value<=final_max_position_value
final_loss_ok = final_return_rate>=-final_max_loss_rate
final_liquidity_ok = final_daily_volume>=final_min_daily_volume
final_market_open = final_market_status.strip().upper()=="OPEN"
final_data_complete = final_risk_timestamp is not None

final_allowed = (
    final_position_ok
    and final_loss_ok
    and final_liquidity_ok
    and final_market_open
    and final_data_complete
)
final_decision = "ALLOW" if final_allowed else "REJECT"

final_report = (
    f"{final_symbol}: value={final_current_value:.2f}, "
    f"net_pnl={final_net_pnl:.2f}, "
    f"return={final_return_rate:.2%}, "
    f"decision={final_decision}"
) 

stress_current_value = final_quantity * stress_price
stress_net_pnl = (
    stress_current_value 
    - final_invested_capital 
    + (final_quantity * final_dividend)
)
stress_return_rate = stress_net_pnl / final_invested_capital

stress_allowed = (
    (stress_current_value <= final_max_position_value)
    and (stress_return_rate >= -final_max_loss_rate)
    and final_liquidity_ok
    and final_market_open
    and final_data_complete
)
stress_decision = "ALLOW" if stress_allowed else "REJECT"
print(final_max_loss_rate)
print(final_report)
print("stress_decision:", stress_decision)


0.04
GAZP: value=61915.00, net_pnl=-1600.40, return=-2.52%, decision=ALLOW
stress_decision: REJECT


## Вывод и защита

В отдельной Markdown-ячейке после этой напишите 5–7 предложений:

1. Почему позиция с отрицательным PnL всё же получила ALLOW?
2. Какой именно лимит нарушен в stress-сценарии?
3. Где в расчётах использовались проценты, а где доли?
4. Какие два исходных значения сильнее всего влияют на решение?
5. Что станет опасным, если данные останутся строками?

### Контрольные вопросы

1. Чем `=` отличается от `==`?
2. Почему `"120" * 3` не равно 360?
3. Зачем делить процент комиссии на 100?
4. Чем `None` отличается от 0 и пустой строки?
5. Почему сравнение float иногда выполняют с допуском?
6. Как приоритет операций влияет на финансовую формулу?
7. Почему риск-проверки полезно хранить отдельно?
8. Что нужно сделать с ноутбуком перед сдачей?
